# DiDAQt FABRIC Experiment

This notebook creates a FABRIC testbed artifact that demonstrates the DiDAQt
fault-detection framework for DAQ networks.

**Topology:**
- 1 Sender node (ConnectX-6, 2 ports) — runs 10 sender instances
- 1 Receiver node (ConnectX-6, 2 ports + 2 NIC_Basic) — runs 2 receiver instances
- 1 Controller node (NIC_Basic) — runs the heartbeat monitor
- 1 Tofino P4 switch — L2 MAC forwarding with VLAN rewriting

**Data path (in-band):** Sender ports ↔ Tofino ↔ Receiver ports (VLAN-tagged)
- Cross-site links use **L2PTP**; same-site links use **L2Bridge**.

**Control path (out-of-band):** Receiver NICs → Controller (FABNetv4 / L3)

## 1. Setup

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager

fablib = FablibManager()

In [ ]:
fablib.list_sites(filter_function=lambda x: x["p4-switch_available"] > 0 and x["nic_connectx_6_available"] > 1, pretty_names=False)

In [ ]:
# ---------- Configuration ----------
# Each component can be placed on a separate FABRIC site.
# Cross-site in-band links use L2PTP; same-site links use L2Bridge.
# Run fablib.list_sites() to find sites with Tofino switches and ConnectX-6 NICs.

SENDER_SITE     = 'UTAH'
RECEIVER_SITE   = 'UTAH'
SWITCH_SITE     = 'UTAH'
CONTROLLER_SITE = 'UTAH'

SLICE_NAME = 'didaqt-experiment'
IMAGE      = 'default_ubuntu_22'

# SDE environment command on FABRIC Tofino nodes (nix-shell wrapper).
# Adjust the version if your site uses a different SDE release.
SDE_ENV = 'sde-env-9.13.3'

## 2. Create Topology

In [ ]:
slice = fablib.new_slice(name=SLICE_NAME)

# ---- Nodes ----
sender_node = slice.add_node(name='sender', site=SENDER_SITE,
                             cores=8, ram=32, disk=20, image=IMAGE)
receiver_node = slice.add_node(name='receiver', site=RECEIVER_SITE,
                               cores=8, ram=32, disk=20, image=IMAGE)
controller_node = slice.add_node(name='controller', site=CONTROLLER_SITE,
                                 cores=4, ram=8, disk=20, image=IMAGE)

# ---- P4 Switch ----
p4_switch = slice.add_switch(name='p4_switch', site=SWITCH_SITE)

# ---- In-band NICs (ConnectX-6, dual-port 100G) ----
sender_nic = sender_node.add_component(model='NIC_ConnectX_6', name='sender_nic')
rx_nic     = receiver_node.add_component(model='NIC_ConnectX_6', name='rx_nic')

sender_ifaces = sender_nic.get_interfaces()
rx_ifaces     = rx_nic.get_interfaces()
sw_ifaces     = p4_switch.get_interfaces()

print(f'Sender NIC interfaces:   {[i.get_name() for i in sender_ifaces]}')
print(f'Receiver NIC interfaces: {[i.get_name() for i in rx_ifaces]}')
print(f'Switch interfaces:       {[i.get_name() for i in sw_ifaces]}')

# ---- In-band L2 networks ----
# Use L2PTP when the two endpoints are on different sites, L2Bridge when same-site.
# Create networks first, then add interfaces (following FABRIC switch pattern).

def l2_type(site_a, site_b):
    return 'L2PTP' if site_a != site_b else 'L2Bridge'

l2_type_sender_sw = l2_type(SENDER_SITE, SWITCH_SITE)
l2_type_rx_sw     = l2_type(RECEIVER_SITE, SWITCH_SITE)

print(f'\nSender  <-> Switch link type: {l2_type_sender_sw}')
print(f'Receiver <-> Switch link type: {l2_type_rx_sw}')

# Create the four L2 networks.
net_s0 = slice.add_l2network(name='net-s0-sw', type=l2_type_sender_sw)
net_s1 = slice.add_l2network(name='net-s1-sw', type=l2_type_sender_sw)
net_r0 = slice.add_l2network(name='net-r0-sw', type=l2_type_rx_sw)
net_r1 = slice.add_l2network(name='net-r1-sw', type=l2_type_rx_sw)

# Add node interfaces (set_mode='config' for post-boot configuration).
for iface, net in [(sender_ifaces[0], net_s0), (sender_ifaces[1], net_s1),
                   (rx_ifaces[0], net_r0), (rx_ifaces[1], net_r1)]:
    iface.set_mode('config')
    net.add_interface(iface)

# Add switch interfaces.
net_s0.add_interface(sw_ifaces[0])
net_s1.add_interface(sw_ifaces[1])
net_r0.add_interface(sw_ifaces[2])
net_r1.add_interface(sw_ifaces[3])

# ---- Out-of-band NICs (NIC_Basic for heartbeat L3 network) ----
rx_ctrl_nic0 = receiver_node.add_component(model='NIC_Basic', name='rx_ctrl0')
rx_ctrl_nic1 = receiver_node.add_component(model='NIC_Basic', name='rx_ctrl1')
ctrl_nic     = controller_node.add_component(model='NIC_Basic', name='ctrl_nic')

ctrl_net = slice.add_l3network(name='ctrl-net', interfaces=[
    rx_ctrl_nic0.get_interfaces()[0],
    rx_ctrl_nic1.get_interfaces()[0],
    ctrl_nic.get_interfaces()[0]
], type='IPv4')

print('\nTopology defined.')
slice.show()

## 3. Submit Slice

In [ ]:
slice.submit()
print('Slice submitted and ready.')

## 4. Gather Topology Information

After the slice is provisioned, query the actual OS interface names,
MAC addresses, VLAN IDs, and L3 IP addresses.

In [ ]:
slice = fablib.get_slice(name=SLICE_NAME)

sender_node     = slice.get_node('sender')
receiver_node   = slice.get_node('receiver')
controller_node = slice.get_node('controller')
p4_switch       = slice.get_node('p4_switch')

def vlan_or_zero(iface):
    """Return the VLAN ID as an int, or 0 if the link has no VLAN (L2Bridge)."""
    v = iface.get_vlan()
    return int(v) if v else 0

# ---- Sender in-band interfaces ----
s_iface0 = sender_node.get_interface(network_name='net-s0-sw')
s_iface1 = sender_node.get_interface(network_name='net-s1-sw')

s0_os   = s_iface0.get_physical_os_interface_name()
s0_mac  = s_iface0.get_mac()
s0_vlan = vlan_or_zero(s_iface0)
s1_os   = s_iface1.get_physical_os_interface_name()
s1_mac  = s_iface1.get_mac()
s1_vlan = vlan_or_zero(s_iface1)

print(f'Sender port 0: iface={s0_os}  mac={s0_mac}  vlan={s0_vlan}')
print(f'Sender port 1: iface={s1_os}  mac={s1_mac}  vlan={s1_vlan}')

# ---- Receiver in-band interfaces ----
r_iface0 = receiver_node.get_interface(network_name='net-r0-sw')
r_iface1 = receiver_node.get_interface(network_name='net-r1-sw')

r0_os   = r_iface0.get_physical_os_interface_name()
r0_mac  = r_iface0.get_mac()
r0_vlan = vlan_or_zero(r_iface0)
r1_os   = r_iface1.get_physical_os_interface_name()
r1_mac  = r_iface1.get_mac()
r1_vlan = vlan_or_zero(r_iface1)

print(f'\nReceiver port 0: iface={r0_os}  mac={r0_mac}  vlan={r0_vlan}')
print(f'Receiver port 1: iface={r1_os}  mac={r1_mac}  vlan={r1_vlan}')

if s0_vlan == 0:
    print('\nNote: VLAN IDs are 0 (L2Bridge / same-site). Frames will be untagged.')
else:
    print(f'\nVLAN tagging is active (L2PTP / cross-site).')

# ---- Controller L3 interface ----
ctrl_iface = controller_node.get_interface(network_name='ctrl-net')
ctrl_ip = ctrl_iface.get_ip_addr()

# The receiver has two NIC_Basic on ctrl-net; get both IPs.
rx_ctrl_ifaces = [i for i in receiver_node.get_interfaces()
                  if i.get_network() and i.get_network().get_name() == 'ctrl-net']
rx_ctrl_ips = [i.get_ip_addr() for i in rx_ctrl_ifaces]

print(f'\nController IP: {ctrl_ip}')
print(f'Receiver ctrl IPs: {rx_ctrl_ips}')

## 5. Install Dependencies

In [ ]:
install_cmd = 'sudo apt-get update -qq && sudo apt-get install -y -qq build-essential ethtool'

from concurrent.futures import ThreadPoolExecutor

def install_on(node):
    name = node.get_name()
    print(f'Installing on {name}...')
    stdout, stderr = node.execute(install_cmd, quiet=True)
    print(f'  {name}: done')
    return stdout, stderr

with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(install_on, n)
               for n in [sender_node, receiver_node, controller_node]]
    for f in futures:
        f.result()

print('All dependencies installed.')

## 6. Upload Source Code

In [ ]:
import os

# Paths relative to this notebook (artifact/)
REPO = os.path.abspath('..')

for node in [controller_node, sender_node, receiver_node]:
    node.upload_directory(REPO, "/home/ubuntu/")

# Switch — only the P4 program
p4_switch.execute(f'mkdir -p examples/p4', quiet=True)
p4_switch.upload_file(
    os.path.join(REPO, 'examples/p4/l2_forward.p4'),
    f'examples/p4/l2_forward.p4'
)
print(f'  p4_switch: P4 program uploaded')

print('Upload complete.')

## 7. Compile

In [ ]:
# Sender: just needs gcc
print('Compiling sender...')
sender_node.execute(
    f'cd {REMOTE_DIR} && make examples',
    quiet=True
)
print('  sender: OK')

# Receiver: needs libdidaqt + receiver
print('Compiling receiver...')
receiver_node.execute(
    f'cd {REMOTE_DIR} && make examples',
    quiet=True
)
print('  receiver: OK')

# Controller: heartbeat monitor
print('Compiling heartbeat_monitor...')
controller_node.execute(
    f'cd {REMOTE_DIR} && make examples',
    quiet=True
)
print('  heartbeat_monitor: OK')

print('All binaries compiled.')

## 8. Configure P4 Switch

Compile the P4 program, load kernel modules, start `bf_switchd`,
and enable the switch ports.

The SDE is accessed via the `sde-env` nix-shell wrapper on FABRIC
Tofino nodes.  All switch commands use interactive prompt-based
execution to handle the nix-shell and bfshell environments.

In [ ]:
P4_SRC = f'{REMOTE_DIR}/examples/p4/l2_forward.p4'
P4_PROG = 'l2_forward'

# Regex that matches the nix-shell prompt (after entering sde-env)
NIX_PROMPT = r'\[nix\-shell.*\$\s*'

# ---- Step 1: Compile P4 program ----
print('Compiling P4 program on switch...')
stdout, stderr = p4_switch.execute(command=[
    (SDE_ENV,                          NIX_PROMPT, 30),
    (f'p4_build.sh {P4_SRC}',         NIX_PROMPT, 120),
    ('exit',                           r'\$\s*',   10),
])
print('  P4 compilation done.')

# Verify build output
stdout, stderr = p4_switch.execute(command=[
    (SDE_ENV,                                                    NIX_PROMPT, 10),
    (f'ls ~/.bf-sde/*/build/{P4_PROG}/tofino/pipe/ 2>/dev/null', NIX_PROMPT, 10),
    ('exit',                                                     r'\$\s*',   10),
])
print(f'  Build artifacts: {stdout.strip()}')

# ---- Step 2: Get logical port names for ucli port-add ----
# get_device_name() returns logical port names like "1", "2", etc.
sw_iface_s0 = p4_switch.get_interface(network_name='net-s0-sw')
sw_iface_s1 = p4_switch.get_interface(network_name='net-s1-sw')
sw_iface_r0 = p4_switch.get_interface(network_name='net-r0-sw')
sw_iface_r1 = p4_switch.get_interface(network_name='net-r1-sw')

sw_port_s0 = sw_iface_s0.get_device_name()
sw_port_s1 = sw_iface_s1.get_device_name()
sw_port_r0 = sw_iface_r0.get_device_name()
sw_port_r1 = sw_iface_r1.get_device_name()

print(f'\n  Switch logical ports: s0={sw_port_s0} s1={sw_port_s1} '
      f'r0={sw_port_r0} r1={sw_port_r1}')

# ---- Step 3: Start bf_switchd, load modules, enable ports ----
# This is a long-running daemon — use execute_thread().
# Port commands use ucli: pm port-add <port>/- <speed> <FEC>
print('\nStarting bf_switchd and enabling ports...')
switch_thread = p4_switch.execute_thread(command=[
    (SDE_ENV,                                                     NIX_PROMPT,  30),
    ('sudo $SDE_INSTALL/bin/./bf_kdrv_mod_load $SDE_INSTALL',     NIX_PROMPT,  20),
    (f'run_switchd.sh -p {P4_PROG}',                              r'bfshell>', 60),
    ('ucli',                                                      r'bf-sde>',  10),
    (f'pm port-add {sw_port_s0}/- 100G NONE',                     r'bf-sde>',  10),
    (f'pm port-add {sw_port_s1}/- 100G NONE',                     r'bf-sde>',  10),
    (f'pm port-add {sw_port_r0}/- 100G NONE',                     r'bf-sde>',  10),
    (f'pm port-add {sw_port_r1}/- 100G NONE',                     r'bf-sde>',  10),
    (f'pm port-enb {sw_port_s0}/-',                               r'bf-sde>',  10),
    (f'pm port-enb {sw_port_s1}/-',                               r'bf-sde>',  10),
    (f'pm port-enb {sw_port_r0}/-',                               r'bf-sde>',  10),
    (f'pm port-enb {sw_port_r1}/-',                               r'bf-sde>',  10),
    ('pm show',                                                   r'bf-sde>',  10),
], output_file='run_switchd.log')

import time
time.sleep(30)  # Wait for switchd to initialize and ports to come up
print('  bf_switchd started and ports enabled.')
print('  (daemon log: run_switchd.log on the switch)')

In [ ]:
# Build and run a bfrt_python script that installs forwarding rules.
# The script dynamically discovers dev_port numbers from the logical
# port names (matching the FABRIC example pattern).

def mac_to_int(mac_str):
    """Convert 'AA:BB:CC:DD:EE:FF' to integer."""
    return int(mac_str.replace(':', ''), 16)

# VLAN IDs on the switch side of each L2 link (0 = untagged / L2Bridge).
sw_vlan_s0 = vlan_or_zero(sw_iface_s0)
sw_vlan_s1 = vlan_or_zero(sw_iface_s1)
sw_vlan_r0 = vlan_or_zero(sw_iface_r0)
sw_vlan_r1 = vlan_or_zero(sw_iface_r1)

# ---- Generate bfrt_python setup script ----
bfrt_script = f'''
# ---- Discover dev_port for each logical port ----
# Logical port names from FABRIC (e.g., "1", "2", "3", "4").
logical_ports = {{
    'r0': '{sw_port_r0}',
    'r1': '{sw_port_r1}',
}}

# Dump port table and parse logical-to-dev_port mapping.
port_dump = bfrt.port.port.dump(return_ents=True, from_hw=True)
port_map = {{}}
if port_dump:
    for ent in port_dump:
        key_fields = ent.key
        data_fields = ent.data
        dp = key_fields.get('$DEV_PORT') or key_fields.get(b'$DEV_PORT')
        pname = data_fields.get('$PORT_NAME') or data_fields.get(b'$PORT_NAME', '')
        if isinstance(pname, bytes):
            pname = pname.decode()
        pname = str(pname)
        connector = pname.split('/')[0] if '/' in pname else pname
        port_map[connector] = dp

print(f"Port mapping: {{port_map}}")

def resolve(name, logical):
    dp = port_map.get(logical)
    if dp is None:
        print(f"WARNING: could not resolve logical port {{logical}} for {{name}}")
        return None
    print(f"  {{name}}: logical={{logical}} -> dev_port={{dp}}")
    return dp

dev_r0 = resolve('receiver0', logical_ports['r0'])
dev_r1 = resolve('receiver1', logical_ports['r1'])

# ---- Install forwarding rules ----
# In TNA, the bfrt tree is: bfrt.<prog>.pipe.<control>.<table>
l2_fwd = bfrt.l2_forward.pipe.Ingress.l2_forward

if dev_r0 is not None:
    l2_fwd.add_with_forward(
        dst_addr={mac_to_int(r0_mac)},
        port=dev_r0,
        vlan_id={sw_vlan_r0}
    )
    print("Rule: dst_mac={r0_mac} -> port={{dev_r0}} vlan={sw_vlan_r0}")

if dev_r1 is not None:
    l2_fwd.add_with_forward(
        dst_addr={mac_to_int(r1_mac)},
        port=dev_r1,
        vlan_id={sw_vlan_r1}
    )
    print("Rule: dst_mac={r1_mac} -> port={{dev_r1}} vlan={sw_vlan_r1}")

bfrt.complete_operations()
l2_fwd.dump(table=True)
'''

print('--- bfrt_python script ---')
print(bfrt_script)

# Upload the script
import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(bfrt_script)
    local_script = f.name

p4_switch.upload_file(local_script, '/tmp/setup_rules.py')
os.unlink(local_script)

# Execute via nix-shell + run_bfshell.sh
print('\nInstalling forwarding rules on switch...')
stdout, stderr = p4_switch.execute(command=[
    (SDE_ENV,                                                         NIX_PROMPT, 10),
    ('run_bfshell.sh --no-status-srv -b /tmp/setup_rules.py',        NIX_PROMPT, 30),
    ('exit',                                                          r'\$\s*',   10),
])
print(stdout)
print('Forwarding rules installed.')

## 9. Configure Node Interfaces

Bring up the in-band (L2) interfaces.  When VLAN tagging is active
(L2PTP / cross-site), VLAN offloading is disabled so the sender and
receiver handle tags in software.
The L3 (FABNetv4) interfaces are auto-configured by FABRIC.

In [ ]:
vlan_active = s0_vlan > 0  # True when L2PTP (cross-site) is in use

# ---- Sender: bring up interfaces ----
for iface, name in [(s0_os, 'sender port 0'), (s1_os, 'sender port 1')]:
    cmds = f'sudo ip link set {iface} up'
    if vlan_active:
        cmds += f' && sudo ethtool -K {iface} txvlan off rxvlan off'
    sender_node.execute(cmds, quiet=True)
    extra = ', VLAN offload disabled' if vlan_active else ''
    print(f'  {name} ({iface}): up{extra}')

# ---- Receiver: bring up in-band interfaces ----
for iface, name in [(r0_os, 'receiver port 0'), (r1_os, 'receiver port 1')]:
    cmds = f'sudo ip link set {iface} up'
    if vlan_active:
        cmds += f' && sudo ethtool -K {iface} txvlan off rxvlan off'
    receiver_node.execute(cmds, quiet=True)
    extra = ', VLAN offload disabled' if vlan_active else ''
    print(f'  {name} ({iface}): up{extra}')

# ---- L3 interfaces are auto-configured by FABRIC ----
# Verify connectivity
print(f'\nVerifying L3 connectivity: receiver -> controller ({ctrl_ip})...')
stdout, stderr = receiver_node.execute(
    f'ping -c 2 -W 2 {ctrl_ip}',
    quiet=True
)
print(stdout)
print('Interface configuration complete.')

## 10. SSH Access

Use these commands to SSH into each node from your local terminal.

In [ ]:
print('='*70)
print('SSH Commands')
print('='*70)
for node in [sender_node, receiver_node, controller_node, p4_switch]:
    name = node.get_name()
    ssh_cmd = node.get_ssh_command()
    print(f'\n--- {name} ---')
    print(f'  {ssh_cmd}')
print()

## 11. Run the Experiment

The cells below print the exact commands to run on each node.
Open SSH sessions to each node (Section 10) and paste these commands.

**Start order:**
1. Controller (heartbeat monitor)
2. Receiver (both instances)
3. Sender (10 instances)

In [ ]:
HB_PORT = 9000

# Build the sender VLAN argument: only pass it when vlan_id > 0.
# With vlan_id=0 (L2Bridge), the sender omits the VLAN tag entirely.
sender_vlan_arg = f' {s0_vlan}' if s0_vlan else ''

print('='*70)
print('STEP 1: Start heartbeat monitor on CONTROLLER')
print('='*70)
print(f'''
cd {REMOTE_DIR}
sudo ./build/heartbeat_monitor {HB_PORT}
''')

print('='*70)
print('STEP 2: Start receiver instances on RECEIVER')
print('='*70)
print(f'''
cd {REMOTE_DIR}

# Receiver instance 0: listens on port 0, heartbeats to controller
sudo ./build/receiver {r0_os} 0 {ctrl_ip} {HB_PORT} &

# Receiver instance 1: listens on port 1, heartbeats to controller
sudo ./build/receiver {r1_os} 1 {ctrl_ip} {HB_PORT} &
''')

print('='*70)
print('STEP 3: Start 10 sender instances on SENDER')
print('='*70)
# All 10 senders use sender port 0, sending to receiver port 0's MAC.
# The switch forwards based on dst_mac to the correct receiver port.
print(f'''
cd {REMOTE_DIR}

# All 10 senders target receiver port 0 via sender port 0.
# Each sender gets a unique sender_id (1-10).
for i in $(seq 1 10); do
    sudo ./build/sender {s0_os} {r0_mac} $i{sender_vlan_arg} &
done

# To stop all senders:
# sudo killall sender
''')

if sender_vlan_arg:
    print(f'(Frames are VLAN-tagged with VID {s0_vlan})')
else:
    print('(Frames are untagged — L2Bridge / same-site)')

print()
print('='*70)
print('EXPECTED OUTPUT')
print('='*70)
print('''
On the controller, you should see heartbeat messages like:

  [14:30:01.234] HB #1 from 10.x.x.x | receiver 0: 10 sender(s) healthy: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  [14:30:01.334] HB #2 from 10.x.x.x | receiver 0: 10 sender(s) healthy: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Receiver 1 will send heartbeats with 0 senders (idle, ready for fail-over):

  [14:30:01.234] HB #3 from 10.x.x.x | receiver 1: 0 sender(s) healthy: [(none)]

On the sender, you should see throughput reports:

  sender 1: 1048576 frames, 9.87 Gbps
  sender 2: 1048576 frames, 9.85 Gbps
  ...
''')

## 12. Cleanup

Delete the slice when you are done with the experiment.

In [ ]:
# Uncomment to delete:
# slice.delete()
# print('Slice deleted.')